<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-09/AI_Modul_9.6_Praktikum_Backpropagation.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 9.6: Praktikum Algoritma Backpropagation & Aturan Rantai Kalkulus
## Mata Kuliah: AI Academy (INSTIPER)

---

### Tujuan Praktikum:
1. Membangun mesin pelatihan jaringan saraf multi-lapis (*Multi-Layer Perceptron*) murni dari dasar menggunakan pustaka NumPy.
2. Mengimplementasikan 4 persamaan fundamental **Backpropagation** secara matriks batch.
3. Membangun modul pengujian kebenaran turunan **Gradient Checking** presisi tinggi ($< 10^{-7}$).
4. Melatih model AI dari nol untuk mengklasifikasikan 4 fraksi kematangan Tandan Buah Segar (TBS) sawit.
5. Memvisualisasikan kurva penurunan rugi dan mengevaluasi sensitivitas laju pembelajaran (*learning rate*).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Pengaturan visualisasi publikasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120

print("[OK] Seluruh pustaka untuk praktikum Backpropagation berhasil dimuat!")

## 1. Implementasi Mesin Pelatihan MLP Murni dari Nol (`MLPScratchEngine`)
Di bawah ini kita mengimplementasikan kelas lengkap yang mencakup perambatan maju, fungsi rugi CCE, perambatan mundur 4 persamaan, dan pembaruan bobot.

In [ ]:
class MLPScratchEngine:
    """
    Mesin Pelatihan Multi-Layer Perceptron Murni (Pure NumPy).
    """
    def __init__(self, layer_dims=[4, 12, 8, 4], learning_rate=0.05, seed=42):
        self.layer_dims = layer_dims
        self.lr = learning_rate
        self.params = {}
        self.loss_history_ = []
        self.acc_history_ = []
        rng = np.random.RandomState(seed)
        
        # Inisialisasi He Normal untuk ReLU dan bias nol
        self.L = len(layer_dims) - 1
        for l in range(1, self.L + 1):
            n_in = layer_dims[l-1]
            n_out = layer_dims[l]
            self.params[f'W{l}'] = rng.normal(0, np.sqrt(2.0 / n_in), size=(n_in, n_out))
            self.params[f'b{l}'] = np.zeros((1, n_out))

    def relu(self, Z):
        return np.maximum(0.0, Z)

    def relu_derivative(self, Z):
        return np.where(Z > 0.0, 1.0, 0.0)

    def softmax(self, Z):
        exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    def forward(self, X):
        cache = {'A0': X}
        A = X
        for l in range(1, self.L):
            Z = np.dot(A, self.params[f'W{l}']) + self.params[f'b{l}']
            A = self.relu(Z)
            cache[f'Z{l}'] = Z
            cache[f'A{l}'] = A
            
        Z_last = np.dot(A, self.params[f'W{self.L}']) + self.params[f'b{self.L}']
        A_last = self.softmax(Z_last)
        cache[f'Z{self.L}'] = Z_last
        cache[f'A{self.L}'] = A_last
        return A_last, cache

    def compute_loss(self, A_last, Y_onehot, eps=1e-15):
        B = Y_onehot.shape[0]
        clipped_A = np.clip(A_last, eps, 1.0 - eps)
        loss = -np.sum(Y_onehot * np.log(clipped_A)) / B
        return loss

    def backward(self, cache, Y_onehot):
        """
        Implementasi 4 Persamaan Fundamental Backpropagation.
        """
        grads = {}
        B = Y_onehot.shape[0]
        
        # Persamaan 1: Galat Lapisan Luaran (Softmax + CCE)
        A_last = cache[f'A{self.L}']
        delta = (A_last - Y_onehot) / B  # Dimensi: (B x K)
        
        # Persamaan 4 & 3 untuk Lapisan Terakhir L
        grads[f'dW{self.L}'] = np.dot(cache[f'A{self.L-1}'].T, delta)
        grads[f'db{self.L}'] = np.sum(delta, axis=0, keepdims=True)
        
        # Loop Perambatan Mundur untuk Lapisan l = L-1 hingga 1
        for l in range(self.L - 1, 0, -1):
            W_next = self.params[f'W{l+1}']
            # Persamaan 2: Propagasi Galat ke Belakang
            delta = np.dot(delta, W_next.T) * self.relu_derivative(cache[f'Z{l}'])
            
            # Persamaan 4 & 3: Gradien Bobot dan Bias
            grads[f'dW{l}'] = np.dot(cache[f'A{l-1}'].T, delta)
            grads[f'db{l}'] = np.sum(delta, axis=0, keepdims=True)
            
        return grads

    def update_parameters(self, grads):
        for l in range(1, self.L + 1):
            self.params[f'W{l}'] -= self.lr * grads[f'dW{l}']
            self.params[f'b{l}'] -= self.lr * grads[f'db{l}']

    def fit(self, X, y, epochs=100, batch_size=32, verbose=True):
        # Konversi y ke One-Hot
        n_classes = self.layer_dims[-1]
        Y_onehot = np.eye(n_classes)[y]
        n_samples = X.shape[0]
        
        self.loss_history_ = []
        self.acc_history_ = []
        
        for epoch in range(1, epochs + 1):
            # Acak sampel data (shuffle)
            perm = np.random.permutation(n_samples)
            X_shuffled = X[perm]
            Y_shuffled = Y_onehot[perm]
            
            # Mini-Batch Loop
            for b in range(0, n_samples, batch_size):
                X_batch = X_shuffled[b:b+batch_size]
                Y_batch = Y_shuffled[b:b+batch_size]
                
                # 1. Forward Pass
                A_last, cache = self.forward(X_batch)
                
                # 2. Backward Pass
                grads = self.backward(cache, Y_batch)
                
                # 3. Update Weights
                self.update_parameters(grads)
                
            # Evaluasi di akhir epoch
            preds_all, _ = self.forward(X)
            epoch_loss = self.compute_loss(preds_all, Y_onehot)
            epoch_acc = np.mean(np.argmax(preds_all, axis=1) == y) * 100
            
            self.loss_history_.append(epoch_loss)
            self.acc_history_.append(epoch_acc)
            
            if verbose and (epoch % 20 == 0 or epoch == 1):
                print(f"Epoch {epoch:3d}/{epochs} - Loss: {epoch_loss:.4f} - Akurasi: {epoch_acc:.2f}%")
                
        return self

    def predict(self, X):
        probs, _ = self.forward(X)
        return np.argmax(probs, axis=1)

print("[OK] Kelas MLPScratchEngine berhasil dibangun secara sempurna!")

## 2. Modul Gradient Checking: Verifikasi Matematika Eksak Beda Hingga
Kita membangun fungsi pengujian independen yang membandingkan gradien analitik dari metode `backward()` terhadap gradien numerik beda hingga dua sisi.

In [ ]:
def gradient_check(model, X_sample, Y_sample_onehot, eps=1e-7):
    """
    Memverifikasi gradien analitik Backprop vs gradien numerik beda hingga.
    """
    # 1. Hitung gradien analitik via Backpropagation
    A_last, cache = model.forward(X_sample)
    analytic_grads = model.backward(cache, Y_sample_onehot)
    
    # 2. Flatten semua parameter dan gradien ke vektor 1D
    param_list = []
    grad_analytic_list = []
    
    for l in range(1, model.L + 1):
        param_list.append(model.params[f'W{l}'].reshape(-1))
        param_list.append(model.params[f'b{l}'].reshape(-1))
        grad_analytic_list.append(analytic_grads[f'dW{l}'].reshape(-1))
        grad_analytic_list.append(analytic_grads[f'db{l}'].reshape(-1))
        
    theta = np.concatenate(param_list)
    grad_analytic = np.concatenate(grad_analytic_list)
    grad_numeric = np.zeros_like(theta)
    
    # 3. Hitung gradien numerik untuk setiap parameter
    # Untuk uji cepat, kita evaluasi 25 parameter acak
    np.random.seed(0)
    sample_indices = np.random.choice(len(theta), size=min(25, len(theta)), replace=False)
    
    for idx in sample_indices:
        # Theta + epsilon
        theta_plus = np.copy(theta)
        theta_plus[idx] += eps
        # Pasang kembali ke model
        _unflatten_params(model, theta_plus)
        A_plus, _ = model.forward(X_sample)
        J_plus = model.compute_loss(A_plus, Y_sample_onehot)
        
        # Theta - epsilon
        theta_minus = np.copy(theta)
        theta_minus[idx] -= eps
        _unflatten_params(model, theta_minus)
        A_minus, _ = model.forward(X_sample)
        J_minus = model.compute_loss(A_minus, Y_sample_onehot)
        
        # Beda hingga dua sisi
        grad_numeric[idx] = (J_plus - J_minus) / (2.0 * eps)
        
    # Kembalikan model ke theta semula
    _unflatten_params(model, theta)
    
    # Hitung selisih relatif pada parameter yang diuji
    g_a = grad_analytic[sample_indices]
    g_n = grad_numeric[sample_indices]
    numerator = np.linalg.norm(g_a - g_n)
    denominator = np.linalg.norm(g_a) + np.linalg.norm(g_n) + 1e-12
    rel_diff = numerator / denominator
    
    return rel_diff

def _unflatten_params(model, theta_vec):
    curr = 0
    for l in range(1, model.L + 1):
        # W
        w_shape = model.params[f'W{l}'].shape
        w_size = w_shape[0] * w_shape[1]
        model.params[f'W{l}'] = theta_vec[curr:curr+w_size].reshape(w_shape)
        curr += w_size
        # b
        b_shape = model.params[f'b{l}'].shape
        b_size = b_shape[1]
        model.params[f'b{l}'] = theta_vec[curr:curr+b_size].reshape(b_shape)
        curr += b_size

print("[OK] Fungsi Gradient Checking berhasil disiapkan!")

## 3. Dataset Agro-Industri: Sortasi Kematangan Tandan Buah Segar (TBS)
Kita membangkitkan 600 sampel data spektral TBS sawit dengan 4 fitur optik terstandardisasi untuk 4 kelas kematangan.

In [ ]:
np.random.seed(42)
n_per_class = 150

# 4 Kelas: Mentah (0), Kurang Matang (1), Matang Sempurna (2), Lewat Matang (3)
f0 = np.random.multivariate_normal([0.20, 0.65, 0.15, 0.01], np.diag([0.003, 0.002, 0.001, 0.0005]), n_per_class)
f1 = np.random.multivariate_normal([0.38, 0.50, 0.35, 0.05], np.diag([0.003, 0.002, 0.002, 0.001]), n_per_class)
f2 = np.random.multivariate_normal([0.58, 0.32, 0.70, 0.18], np.diag([0.003, 0.002, 0.002, 0.002]), n_per_class)
f3 = np.random.multivariate_normal([0.48, 0.25, 0.45, 0.45], np.diag([0.004, 0.002, 0.003, 0.003]), n_per_class)

X_all = np.vstack([f0, f1, f2, f3])
y_all = np.concatenate([np.full(n_per_class, 0), np.full(n_per_class, 1), np.full(n_per_class, 2), np.full(n_per_class, 3)])

# Split Train & Test
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.25, random_state=42, stratify=y_all)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset TBS Sawit: {len(X_train_scaled)} sampel latih, {len(X_test_scaled)} sampel uji.")

## 4. Eksekusi Uji Gradient Checking pada Model Riil
Sebelum melatih model, kita buktikan secara matematis bahwa gradien analitik `backward()` memiliki presisi tinggi.

In [ ]:
test_model = MLPScratchEngine(layer_dims=[4, 8, 6, 4], seed=12)
sample_X = X_train_scaled[:8]
sample_Y = np.eye(4)[y_train[:8]]

diff = gradient_check(test_model, sample_X, sample_Y)
print(f"=== HASIL EVALUASI GRADIENT CHECKING ===")
print(f"Perbedaan Relatif (Relative Difference): {diff:.2e}")
if diff < 1e-6:
    print("[STATUS: SUKSES 100%] Implementasi Backpropagation terbukti benar secara matematis!")
else:
    print("[STATUS: PERINGATAN] Nilai selisih relatif melebihi ambang batas toleransi!")

## 5. Pelatihan Model Penuh dari Awal (*Full Training Loop*)
Kita melatih model dengan arsitektur `4 -> 12 -> 8 -> 4` selama 80 epoch menggunakan laju pembelajaran $\eta = 0.05$.

In [ ]:
mlp_tbs = MLPScratchEngine(layer_dims=[4, 12, 8, 4], learning_rate=0.06, seed=42)
mlp_tbs.fit(X_train_scaled, y_train, epochs=80, batch_size=32, verbose=True)

y_pred_test = mlp_tbs.predict(X_test_scaled)
test_acc = accuracy_score(y_test, y_pred_test)
print(f"\nAkurasi Akhir pada Data Uji Independen: {test_acc * 100:.2f}%")

## 6. Visualisasi Kurva Konvergensi Pembelajaran
Kita memetakan penurunan fungsi rugi dan kenaikan akurasi sepanjang 80 epoch untuk memverifikasi kestabilan penurunan gradien.

In [ ]:
epochs_arr = range(1, len(mlp_tbs.loss_history_) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))

# Kurva Loss
ax1.plot(epochs_arr, mlp_tbs.loss_history_, color='#e74c3c', lw=2.2)
ax1.set_title('Penurunan Nilai Rugi (Categorical Cross-Entropy)', fontweight='bold')
ax1.set_xlabel('Epoch Pelatihan')
ax1.set_ylabel('Loss CCE')
ax1.grid(True, linestyle='--', alpha=0.6)

# Kurva Akurasi
ax2.plot(epochs_arr, mlp_tbs.acc_history_, color='#27ae60', lw=2.2)
ax2.set_title('Peningkatan Akurasi Pelatihan (%)', fontweight='bold')
ax2.set_xlabel('Epoch Pelatihan')
ax2.set_ylabel('Akurasi (%)')
ax2.set_ylim(20, 105)
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 7. Evaluasi Mutu Sortasi: Confusion Matrix
Kita memverifikasi hasil pemilahan fraksi TBS sawit untuk memastikan tidak ada kesalahan sortasi fatal.

In [ ]:
classes = ['Mentah', 'Kurang Matang', 'Matang Sempurna', 'Lewat Matang']

cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix Sortasi TBS (Backprop dari Dasar)', fontweight='bold', pad=12)
plt.xlabel('Prediksi Model')
plt.ylabel('Status Aktual Lapangan')
plt.show()

print("Laporan Lengkap Metrik Klasifikasi Mutu:")
print(classification_report(y_test, y_pred_test, target_names=classes))